# 00a - Build the labelled domain universe

Turns the raw downloads in `data/00_raw/` into two files everything else depends on:

- `data/02_interim/domains_labelled.parquet` - the full labelled corpus
- `data/02_interim/probe_universe.parquet` - the stratified subset to probe

**Download these into `00_raw/` first.** Every file keeps its snapshot identity in the filename: the Tranco list ID, and the capture date for any live feed. That filename is the reproducibility citation.

| Source | Access | Note |
|---|---|---|
| Tranco | direct | record the permanent list ID |
| UMUDGA | direct | one file per DGA family |
| DGArchive | apply to Fraunhofer FKIE | **weeks** - apply today |
| CIC-Bell-DNS2021 | request from UNB | a day or two |
| PhishTank / OpenPhish / URLhaus | direct | live feeds - date-stamp the snapshot |

If DGArchive has not arrived, UMUDGA alone is workable; family coverage is narrower, so say so in the paper rather than pretending otherwise.

In [ ]:
# --- standard header ---
from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/secure-dns-trust-ai'
!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/sandesh20lamichhane/secure-dns-trust-ai.git {REPO}
import sys, os; sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'
%load_ext autoreload
%autoreload 2
from src.utils import config, manifest, seeds, io
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
from pathlib import Path
RAW = Path(P['data']['raw'])
for f in sorted(RAW.rglob('*')):
    if f.is_file():
        print(f'{f.relative_to(RAW)}  ({f.stat().st_size/1e6:.1f} MB)')

## Load each source

Each loader normalises to `domain | label | family | source | first_seen`. Comment out any source you don't have yet - the assembly below tolerates missing pieces.

In [ ]:
from src.data import universe as U
frames = []

# --- benign ---
frames.append(U.load_tranco(RAW/'tranco_XXXXX.csv', top_n=300_000))

# --- malicious ---
frames.append(U.load_umudga(RAW/'umudga'))
# frames.append(U.load_dgarchive(RAW/'dgarchive_full.csv'))
# frames.append(U.load_cic_bell(RAW/'cic_bell_dns2021'/'CSV'/'all.csv'))
frames.append(U.load_phish_feed(str(RAW/'phishtank_2026-08-23.csv'), 'phishtank'))
# frames.append(U.load_phish_feed(str(RAW/'urlhaus_2026-08-23.csv'), 'urlhaus', 'malware'))

for f in frames:
    print(f['source'].iloc[0], len(f), '| families:', f['family'].nunique())

In [ ]:
df = U.combine(frames)
print('total', len(df))
print('label balance:', df['label'].value_counts(normalize=True).round(4).to_dict())
print('imbalance ratio 1:%d' % round((df.label==0).sum()/max((df.label==1).sum(),1)))
print('label conflicts dropped:', df.attrs.get('n_conflicts_dropped'))
df.groupby('source')['label'].agg(['count','mean'])

### Family coverage

The family-disjoint split needs enough distinct families to hold some out. Fewer than ~10 usable families makes that split unstable - report it as a limitation rather than quietly falling back to random splitting.

In [ ]:
fam = df[df.label==1]['family'].value_counts()
print('distinct families:', len(fam))
print('families with >=500 domains:', (fam>=500).sum())
fam.head(30)

In [ ]:
out = Path(P['data']['interim']); out.mkdir(parents=True, exist_ok=True)
df.to_parquet(out/'domains_labelled.parquet', index=False, compression='zstd')
print('wrote', out/'domains_labelled.parquet', df.shape)

## Probe universe

The certificate probe is the expensive step, so it runs on a stratified subset. Malicious rows are sampled per family so no single large family dominates.

50k domains at ~100 concurrent probes with a 10s timeout is roughly 2-4 hours of wall clock, assuming a typical failure rate. Start with 50k; widen later if coverage turns out lower than expected.

In [ ]:
probe = U.probe_universe(df, n_total=50_000, malicious_fraction=0.30, seed=42)
probe.to_parquet(out/'probe_universe.parquet', index=False, compression='zstd')
print(len(probe), probe['label'].value_counts().to_dict())
print('families in probe set:', probe[probe.label==1]['family'].nunique())

Next: open `02_certificate_collection.ipynb` and start it. Leave it running.

While it runs, work through `01_data_audit`, `04_split_creation`, and baselines on lexical + DNS features - none of those need certificate data.